In [30]:
import learnQ as lq
import morphData as morph
import cvxpy as cp
import pandas as pd
import torch


## Solving for the weights using brute force approach

There are two "brute force" options:
1. Solve for the optimal weights at the final timepoint before treatment, $T$.
2. Solve for the optimal weights that achieve balance over all pre-treatment timepoints.

Let $Y_{itk}$ represent the value of outcome $k$ for unit $i$ at time $t$. Denote the vector of outcomes at time $t$ for unit $i$ by $\vec{Y}_{it}$. Denote the $(N-1,K)$ matrix of donor unit outcomes at time $t$ by $\mathbf{Y}_t$. Each column in this matrix is a vector $\vec{Y}_{it}$ for $i\in 2,\ldots,N$.
The weights for option (1) are found by solving:
$$
\argmin_{\vec{w}\in \mathbb{R}^{N-1}} ||Y_{1,T} - \mathbf{Y}_{T}\vec{w}|| \quad\text{subject to}\quad \sum_{i=1}^{N-1} w_i = 1 \quad\text{and}\quad w_i\geq 0,
$$ 
which is the quadratic program:
$$
\argmin_{w\in\mathbb{R}^{N-1}} \vec{w}^T\mathbf{Y}_T^{\top}Y_T\vec{w} + (- Y_{1T}^\top\mathbf{Y}_T)\vec{w} \quad\text{subject to}\quad \sum_{i=1}^{N-1} w_i = 1 \quad\text{and}\quad w_i\geq 0
$$
The bias of interest is simply the discrepancy at time $T+1$. Crucially, this is a poor way of constructing a synthetic control since it leverages only the information from time $T$.
Now, let $\vec{Y}_1$ be the vector of all (pre-treatment) outcomes for unit 1, consisting of the vectors $\vec{Y}_{1t}$ for $t\in 1,\ldots, T$ stacked. Similarly, let $\mathbf{Y}$ be the stacked matrices $\mathbf{Y}_t$ for $t\in 1,\ldots T$. 
The weights for option (2) are found by solving:
$$
\argmin_{w\in\mathbb{R}^{N-1}} ||Y_{1} - \mathbf{Y}\vec{w}|| \quad\text{subject to}\quad \sum_{i=1}^{N-1} w_i = 1 \quad\text{and}\quad w_i\geq 0,
$$ 
which is easily seen to be a QP as above. Though this estimate leverages all pre-treatment information, the hypothesis is that the problem of finding optimal weights is over constrained. We expect this to result in high bias of the estimate of $\vec{Y}_{1,T+1}$.


In [31]:
# Read in the data
treated_data = pd.read_csv('treated_data.csv')
control_data = pd.read_csv('control_data.csv')


## First brute force approach:

In [32]:
# Using the penultimate time point to fit a synthetic control

# This approach leverages the most recent data available before the intervention - it doesn't provide much of a synthetic control, since it simply uses
# the nearest observed patient to the target patient as the SC.

def brute_force_1(t, treated_data, control_data):
    train_target_vectors, train_covariate_matrices, test_target_vector, test_covariate_matrix = morph.morph(treated_data, control_data)

    Y_1T = train_target_vectors[t]
    print("The shape of the target vector is: ",Y_1T.shape)
    Y_T = train_covariate_matrices[t]
    print("The shape of the covariates is: ",Y_T.shape)

    N_minus_1 = Y_T.shape[1]
    print("There are: ",N_minus_1," weights being estimated")
    w = cp.Variable(N_minus_1, nonneg=True) # Enforces w >= 0

    # Enforce the sum == 1 constraint
    constraints = [cp.sum(w) == 1]

    # Objective and problem definition
    obj = cp.Minimize(cp.sum_squares(Y_1T - Y_T @ w))
    prob = cp.Problem(obj, constraints)
    prob.solve()

    # Print the results
    print("The estimated weights are: \n",(w.value).round(2))
    print("The discrepancy is:\n",torch.sum((test_target_vector - test_covariate_matrix @ w.value)**2).item())
    print("The estimated outcomes are:\n", (test_covariate_matrix @ w.value).numpy().round(2))
    print("The actual outcomes are:\n", test_target_vector.numpy().round(2))

brute_force_1(38, treated_data, control_data)

The shape of the target vector is:  torch.Size([10])
The shape of the covariates is:  torch.Size([10, 49])
There are:  49  weights being estimated
The estimated weights are: 
 [0.43 0.16 0.05 0.   0.   0.18 0.   0.17 0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.  ]
The discrepancy is:
 12.738452691133684
The estimated outcomes are:
 [4.51 4.49 4.14 5.33 4.32 3.76 4.16 4.79 4.49 3.89]
The actual outcomes are:
 [3.91 6.   5.06 5.25 3.62 5.79 6.03 5.63 5.08 3.75]


## Second Brute Force Approach

In [33]:
# a list of tensors - each one is a vector of outcomes. There are T many of these, one for each time point.

def brute_force_2(t, treated_data, control_data):
    train_target_vectors, train_covariate_matrices, test_target_vector, test_covariate_matrix = morph.morph(treated_data, control_data)

    Y_1 = train_target_vectors[0:t]
    Y = train_covariate_matrices[0:t]

    N_minus_1 = Y[0].shape[1]

    print("There are: ",N_minus_1," weights being estimated")
    w = cp.Variable(N_minus_1, nonneg=True) # Enforces w >= 0

    # Enforce the sum == 1 constraint
    constraints = [cp.sum(w) == 1]

    # Objective and problem definition
    obj = cp.Minimize(sum(cp.sum_squares(target - covariates @ w) for target, covariates in zip(Y_1, Y)))
    prob = cp.Problem(obj, constraints)
    prob.solve()

    print("The estimated weights are: \n",(w.value).round(2))
    print("The discrepancy is:\n", torch.sum((test_target_vector - test_covariate_matrix @ w.value)**2).item())
    print("The estimated outcomes are:\n", (test_covariate_matrix @ w.value).numpy().round(2))
    print("The actual outcomes are:\n", test_target_vector.numpy().round(2))

brute_force_2(38, treated_data, control_data)


There are:  49  weights being estimated
The estimated weights are: 
 [0.15 0.12 0.12 0.15 0.14 0.03 0.02 0.12 0.03 0.05 0.04 0.   0.   0.02
 0.02 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.  ]
The discrepancy is:
 12.887516320187776
The estimated outcomes are:
 [4.22 4.3  4.32 4.76 4.71 4.38 3.9  5.1  4.32 4.46]
The actual outcomes are:
 [3.91 6.   5.06 5.25 3.62 5.79 6.03 5.63 5.08 3.75]


## How well do the methods capture long range effects? 

1. Fit brute force approach 1 on data at time $t=19$ (first $20$ time periods).
2. Fit brute force approach 2, LearnQ, and EBM method on first $20$ time periods.
3. test discrepancy at final time point $T$.

In [34]:
# Fitting the first brute force approaches using restricted data
# and testing, still on the last time point.

t = 19 # the middle time point
brute_force_1(t, treated_data, control_data) # train on only t = 19
brute_force_2(t, treated_data, control_data) # train *up to* t = 19



The shape of the target vector is:  torch.Size([10])
The shape of the covariates is:  torch.Size([10, 49])
There are:  49  weights being estimated
The estimated weights are: 
 [0.   0.   0.   0.   0.   0.   0.   0.48 0.   0.11 0.   0.38 0.   0.03
 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.  ]
The discrepancy is:
 21.927033186155647
The estimated outcomes are:
 [4.06 3.15 4.79 5.31 4.04 3.94 3.79 3.64 4.07 4.01]
The actual outcomes are:
 [3.91 6.   5.06 5.25 3.62 5.79 6.03 5.63 5.08 3.75]
There are:  49  weights being estimated
The estimated weights are: 
 [0.11 0.17 0.08 0.12 0.21 0.03 0.   0.03 0.05 0.   0.04 0.   0.   0.08
 0.07 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
 0.   0.   0.   0.   0.   0.   0.  ]
The discrepancy is:
 15.5472026643118
The estimated outcomes are